In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import ThomasFermi

From notes Chris made in paper: 

The single-electron states are of the form
$\begin{equation}
\psi_{nkm}(r,\theta,\phi) = \frac1{r\sqrt{\sin\theta}}[F_{nk}(r)\Theta^{(F)}_{km}(\theta,\phi) + G_{nk}(r) \Theta^{(G)}_{km}(\theta,\phi)]
\end{equation}$

where the radial functions satisfy (with units put back in)

$\begin{equation}
\left( \begin{array}{cc}
m_ec^2+q_eV & \hbar c(\frac kr + \partial_r) \\
\hbar c(\frac kr - \partial_r) & -m_ec^2+q_eV
\end{array} \right)
\left(\begin{array}{c} F_{nk}(r) \\ G_{nk}(r) \end{array}\right)
=
h_{nk}\left(\begin{array}{c} F_{nk}(r) \\ G_{nk}(r) \end{array}\right).
\end{equation}$

reordered to be a bit clearer 
$\begin{eqnarray}
-\frac{m_e c^2 + q_e V -h_{nk}}{\hbar c}F_{nk}(r) -  (\frac{k}{r})G_{nk}(r) &=& \partial_r G_{nk}(r) \\
\frac{k}{r} F_{nk}(r) + \frac{-m_e c^2 +q_e V - h_{nk}}{\hbar c }G_{nk}(r) & =& \partial_r F_{nk}(r)
\end{eqnarray}$

In [ ]:
r_bohr_H = 5.2946541e-11  # bohr radius in meters
r_bohr_Mg = r_bohr_H / 80
r_min = r_bohr_Mg / 3
print(r_min)

In [ ]:
a_from_paper = 6.468e-10

# r_max = (3*vol_per_atom/(4*np.pi))**(1/3)
r_max = a_from_paper

In [ ]:
r_array = np.logspace(-13, -9.2, 100)
pot_Hg = np.zeros(len(r_array))
pot_Cd = np.zeros(len(r_array))
pot_Te = np.zeros(len(r_array))
for i in range(len(r_array)):
    pot_Hg[i] = ThomasFermi.Potential(80, r_array[i])
    pot_Cd[i] = ThomasFermi.Potential(52, r_array[i])
    pot_Te[i] = ThomasFermi.Potential(48, r_array[i])

In [ ]:
plt.loglog(r_array, pot_Hg)
plt.loglog(r_array, pot_Cd)
plt.loglog(r_array, pot_Te)
plt.show()

In [ ]:
class ElectronWaveFunctionGRC:

    """

    Electron Wave Function class implements the parameters and RK-4 method for the electron's radial part of the
    wave function.


    Parameters
    ----------

    nu: float

    k: float
       quantum number

    h: float
       Energy eigenvalue


    mu: float
        mass of the electron

    """

    def __init__(self, n, k, h, tol, Z):
        self.n = n
        self.k = k
        self.h = h
        self.mu = 9.1093837e-31  # kilograms #define in right units here
        self.c = 299792458.0  # m/s  #define in right units here
        self.hbar = 1.054571817e-34  # Js #define in right units here
        self.charge = -1.60217663e-19  # V
        self.alpha = 0.0072973525643  # alpha
        self.Z = Z  # define this
        self.tol = tol

        self.nu = np.sqrt(self.k**2 - (self.alpha * self.Z) ** 2)
        # print(self.nu,(self.alpha*self.Z)**2)
        self.coeff = None

    def potential(self, r):
        return ThomasFermi.Potential(self.Z, r)

    def diff_eq(self, r, F, G):
        dG = (self.h - self.mu * self.c**2 - self.charge * self.potential(r)) * F / self.hbar / self.c - (
            self.k / r
        ) * G

        dF = (self.k / r) * F + (
            -self.h - self.mu * self.c**2 + self.charge * self.potential(r)
        ) * G / self.hbar / self.c

        return np.asarray([dF, dG])

    def RK_4(self, r_min, r_inf, N):
        # this is in terms of r_star

        step_size = (r_inf - r_min) / (N - 1)
        # print(step_size)

        r_points = np.linspace(r_min, r_inf, N)

        # print(r_points[0])

        # Define initial values for F and G
        # inner boundary condition
        F = np.sqrt(self.k**2 - self.nu**2)
        G = self.k - self.nu

        # print(F,G)
        # outer boundary condition is F = G

        # Arrays to keep track of r, F and G points:

        F_points = []
        G_points = []
        K_counts = []
        counter_K = 0
        K_check = 2.0**60

        for r in r_points:
            # Update F_points and G_points arrays:
            F_points.append(F)
            G_points.append(G)
            K_counts.append(counter_K)

            # Calculate all the slopes (each variable has a separate k-slope):
            (k1F, k1G) = tuple(step_size * self.diff_eq(r, F, G))

            (k2F, k2G) = tuple(step_size * self.diff_eq(r + 0.5 * step_size, F + 0.5 * k1F, G + 0.5 * k1G))

            (k3F, k3G) = tuple(step_size * self.diff_eq(r + 0.5 * step_size, F + 0.5 * k2F, G + 0.5 * k2G))

            (k4F, k4G) = tuple(step_size * self.diff_eq(r + step_size, F + k3F, G + k3G))

            # Calculate next F_point and G_point
            F = F + (k1F + 2 * k2F + 2 * k3F + k4F) / 6
            G = G + (k1G + 2 * k2G + 2 * k3G + k4G) / 6
            if F**2 + G**2 > K_check:
                # print('true')
                F = F / np.sqrt(K_check)
                G = G / np.sqrt(K_check)
                counter_K += 1
            # K_counts.append(counter_K)
        # want to normalize the last points of things

        coeff = (
            np.sum(np.abs(F_points[1:-1])) ** 2
            + np.sum(np.abs(G_points[1:-1]) ** 2)
            + 0.5
            * (
                np.abs(F_points[0]) ** 2
                + np.abs(F_points[-1]) ** 2
                + np.abs(G_points[0]) ** 2
                + np.abs(G_points[-1] ** 2)
            )
        ) * step_size

        self.coeff = coeff
        # print('coeff is ' + str(coeff))
        # F_points= F_points/coeff
        # G_points= G_points/coeff
        return np.array(r_points), np.array(F_points), np.array(G_points), coeff, K_counts

In [ ]:
n = 1
k = 1
# tol = 1e-10
Nr = 3000
hmin = 0.5 * 1.60218e-19  # band gap energy eV to J
hmax = 10e6 * 1.60218e-19  # takes me up to a MeV for now to J
h = np.sqrt(hmin * hmax)
s = (hmax / hmin) ** 0.25
# print(h,s)
tested_hs = [h]
tested_s = [s]
n_zeros = [0]
# Z=80


def iterate_for_zeros(h, s, n, k, Z, hsarray, sarray, nzeros):
    # number of iterations is limited by how big s can be
    # and the number of bits stored in float64
    for _ in range(53 + max(int(np.ceil(np.log2(s))), 0)):
        testing = ElectronWaveFunctionGRC(n, k, h, tol, Z)
        temp_data_rs, temp_data_Fs, temp_data_Gs, coeff, K_count = testing.RK_4(r_min, r_max, Nr)
        # print(temp_data_rs[0], temp_data_Fs[0], temp_data_Gs[0])
        check_array = temp_data_Fs - temp_data_Gs
        # print(check_array)
        crossing_finder = check_array[:-1] * check_array[1:]
        num_zeros = (crossing_finder < 0).sum()
        nzeros.append(num_zeros)
        # print(h,s,num_zeros)
        if num_zeros >= n:
            h = h / s
            hsarray.append(h)
        else:
            h = h * s
            hsarray.append(h)
        s = np.sqrt(s)
        sarray.append(s)
    return {
        "rs": temp_data_rs,
        "Fs_int_out": temp_data_Fs,
        "Gs_int_out": temp_data_Gs,
        "coeff_out": coeff,
        "K_count": K_count,
    }

In [ ]:
data = iterate_for_zeros(h, s, n, k, Z, tested_hs, tested_s, n_zeros)

In [ ]:
n2 = 1
k2 = 1
# tol = 1e-10
h2 = np.sqrt(hmin * hmax)
s2 = (hmax / hmin) ** 0.25
print(h2, s2)
tested_hs2 = [h2]
tested_s2 = [s2]
n_zeros2 = [0]
data2 = iterate_for_zeros(h2, s2, n2, k2, Z, tested_hs2, tested_s2, n_zeros2)

In [ ]:
6.855911732937742e-14 / 6.855908991793074e-14

In [ ]:
tested_hs, tested_s

In [ ]:
# plt.plot(data[0],data[1])
# plt.plot(data[0],data[2])
plt.plot(data2["rs"], np.abs(data2["Fs_int_out"]))
plt.plot(data2["rs"], np.abs(data2["Gs_int_out"]))

plt.xscale("log")
# plt.yscale('symlog')
plt.show()

# plt.plot(data[0],data[4])
plt.plot(data2["rs"], data2["K_count"])
plt.xscale("log")
# plt.yscale('symlog')
plt.show()

In [ ]:
np.array(tested_hs) / 1.60218e-19

In [ ]:
np.array(tested_hs2) / 1.60218e-19

In [ ]:
(-tested_hs[-1] + tested_hs2[-1]) / 1.60218e-19

In [ ]:
def test_transitions(Z, n1, k1, n2, k2):
    tested_h = [h]
    tested_s = [s]
    n_zeros = [0]
    data2 = iterate_for_zeros(h, s, n1, k1, Z, tested_hs, tested_s, n_zeros)

    tested_hs2 = [h2]
    tested_s2 = [s2]
    n_zeros2 = [0]
    data2 = iterate_for_zeros(h2, s2, n2, k2, Z, tested_hs2, tested_s2, n_zeros2)
    print(tested_hs[-1] / 1.60218e-19, tested_hs2[-1] / 1.60218e-19)
    return (-tested_hs[-1] + tested_hs2[-1]) / 1.60218e-19

In [ ]:
energy_gap = test_transitions(80, 1, 1, 2, 1)

In [ ]:
print(energy_gap / 68260.5)  # seems to be close? KL1

In [ ]:
energy_gap2 = test_transitions(80, 1, 1, 1, -1)

In [ ]:
print(energy_gap2 / 68895.1)  # KL2

In [ ]:
energy_gap3 = test_transitions(80, 1, 1, 1, 2)  # KL3

In [ ]:
print(energy_gap3 / 70819.5)

In [ ]:
energy_gap4 = test_transitions(80, 1, 1, 3, 1)

In [ ]:
print(energy_gap4 / 79541.6)  # KM1

In [ ]:
energy_gap_te = test_transitions(52, 1, 1, 2, 1)

In [ ]:
energy_gap_te / 26875.7

In [ ]:
energy_gap_te2 = test_transitions(52, 1, 1, 1, -1)

In [ ]:
energy_gap_te2 / 27201.73

In [ ]:
energy_gap_test = test_transitions(80, 1, 2, 1, 4)  # L3N7

In [ ]:
energy_gap_test / 12184.7  # /12180.6

### ok officially have a good energy finder, now want to calcuate the wavefunctions and get around the rescaling numerical issue. 

In [ ]:
"""def wavefunc_renorm(Z,n,k,bound):
    hmin = .5 * 1.60218e-19 #band gap energy eV to J
    hmax = 10e6* 1.60218e-19 #takes me up to a MeV for now to J 
    h = np.sqrt(hmin*hmax)
    s = (hmax/hmin)**.25
    

    tested_hs = [h]
    tested_s = [s]
    n_zeros = [0]
    data = iterate_for_zeros(h,s,n,k,Z,tested_hs,tested_s,n_zeros)
    h_found = tested_hs[-1]
    print(h_found)
    rescale_factor = 1
    #print(data['K_count'])
    
    data['Fs_int_out'] = np.array(data['Fs_int_out'])*((2.**60)**(np.array(data['K_count'])/2))
    data['Gs_int_out'] = np.array(data['Gs_int_out'])*((2.**60)**(np.array(data['K_count'])/2))
    final_fs = data['Fs_int_out']
    final_gs = data['Gs_int_out']
    
    if bound ==True:

        fout_prime = (data['Fs_int_out'][:-1]-data['Fs_int_out'][1:])/(data['rs'][:-1]-data['rs'][1:])
        crossings = np.array(np.where(fout_prime[:-1]*fout_prime[1:] <0))
        #print(crossings[0])
        # revisit rmatch_index for only 1 crossing
        # rmatch_index = int(np.floor((crossings[0,0]+crossings[0,1])/2)+1)
        rmatch_index = int(np.floor(crossings[0,0])+1)
        print(rmatch_index)

        instance2 = ElectronWaveFunctionGRC(n, k, h_found, tol,Z)
        temp_data_rs, temp_data_Fs, temp_data_Gs,coeff,K_count = instance2.RK_4(r_max,data['rs'][rmatch_index], Nr-rmatch_index)
        
        temp_data_Fs = temp_data_Fs*((2.**60)**(np.array(K_count)/2))
        temp_data_Gs = temp_data_Gs*((2.**60)**(np.array(K_count)/2))
        rescale_factor =  data['Fs_int_out'][rmatch_index]/temp_data_Fs[-1]
        
        print(rmatch_index, data['rs'][rmatch_index], rescale_factor)
        print(data['Fs_int_out'][rmatch_index-1:rmatch_index+2])
        print(data['Fs_int_out'][rmatch_index]/data['Fs_int_out'][rmatch_index+1]*temp_data_Fs[-2]/temp_data_Fs[-1])
        print(temp_data_Fs)
        #print(K_count)
        final_fs[rmatch_index:] = temp_data_Fs[::-1]/rescale_factor
        final_gs[rmatch_index:] = temp_data_Gs[::-1]/rescale_factor
        
    coeff = (np.sum(np.abs(final_fs[1:-1]))**2 + np.sum(np.abs(final_gs[1:-1])**2) + .5*(np.abs(final_fs[0])**2+np.abs(final_fs[-1])**2+ np.abs(final_gs[0])**2+np.abs(final_gs[-1]**2 )))*(data['rs'][1] - data['rs'][0]) 

        #temp_data_rs, Fs_int_in, Gs_int_in,coeff,K_count_in =  instance2.RK_4(r_max,r_min, Nr)
    return data['rs'],final_fs, final_gs,coeff"""

In [ ]:
def wavefunc_renorm2(Z, n, k, bound):
    hmin = 0.5 * 1.60218e-19  # band gap energy eV to J
    hmax = 10e6 * 1.60218e-19  # takes me up to a MeV for now to J
    h = np.sqrt(hmin * hmax)
    s = (hmax / hmin) ** 0.25

    tested_hs = [h]
    tested_s = [s]
    n_zeros = [0]
    data = iterate_for_zeros(h, s, n, k, Z, tested_hs, tested_s, n_zeros)
    h_found = tested_hs[-1]
    print(h_found)
    rescale_factor = 1
    # print(data['K_count'])

    data["Fs_int_out"] = np.array(data["Fs_int_out"])  # *((2.**60)**(np.array(data['K_count'])/2))
    data["Gs_int_out"] = np.array(data["Gs_int_out"])  # *((2.**60)**(np.array(data['K_count'])/2))
    final_fs = data["Fs_int_out"]
    final_gs = data["Gs_int_out"]
    delta_K = np.array(data["K_count"][:-1]) - np.array(data["K_count"][1:])
    print(final_fs[0:6])
    print(data["K_count"][:6])
    if bound == True:
        fout_prime = (data["Fs_int_out"][:-1] * (2**60) ** (delta_K / 2) - data["Fs_int_out"][1:]) / (
            data["rs"][:-1] - data["rs"][1:]
        )
        crossings = np.array(np.where(fout_prime[:-1] * fout_prime[1:] < 0))
        print(crossings[0])
        # revisit rmatch_index for only 1 crossing
        # rmatch_index = int(np.floor((crossings[0,0]+crossings[0,1])/2)+1)
        rmatch_index = int(np.floor(crossings[0, 0]) + 1)
        print(rmatch_index)

        instance2 = ElectronWaveFunctionGRC(n, k, h_found, tol, Z)
        temp_data_rs, temp_data_Fs, temp_data_Gs, coeff, K_count = instance2.RK_4(
            r_max, data["rs"][rmatch_index], Nr - rmatch_index
        )
        dK_count = np.array(K_count) - K_count[-1]

        temp_data_Fs = temp_data_Fs * ((2.0**60) ** (dK_count / 2))
        temp_data_Gs = temp_data_Gs * ((2.0**60) ** (dK_count / 2))
        rescale_factor = data["Fs_int_out"][rmatch_index] / temp_data_Fs[-1]

        print(rmatch_index, data["rs"][rmatch_index], rescale_factor)
        print(data["Fs_int_out"][rmatch_index - 1 : rmatch_index + 2])
        print(
            data["Fs_int_out"][rmatch_index]
            / data["Fs_int_out"][rmatch_index + 1]
            * temp_data_Fs[-2]
            / temp_data_Fs[-1]
        )
        print(temp_data_Fs)
        # print(K_count)
        final_fs[rmatch_index:] = temp_data_Fs[::-1] * rescale_factor
        final_gs[rmatch_index:] = temp_data_Gs[::-1] * rescale_factor

        fwd_dK_count = np.array(data["K_count"])[:rmatch_index] - data["K_count"][rmatch_index]
        print(fwd_dK_count)
        final_fs[:rmatch_index] *= (2.0**60) ** (fwd_dK_count / 2)
        final_gs[:rmatch_index] *= (2.0**60) ** (fwd_dK_count / 2)

    coeff = (
        np.sum(np.abs(final_fs[1:-1])) ** 2
        + np.sum(np.abs(final_gs[1:-1]) ** 2)
        + 0.5
        * (
            np.abs(final_fs[0]) ** 2
            + np.abs(final_fs[-1]) ** 2
            + np.abs(final_gs[0]) ** 2
            + np.abs(final_gs[-1] ** 2)
        )
    ) * (data["rs"][1] - data["rs"][0])

    # temp_data_rs, Fs_int_in, Gs_int_in,coeff,K_count_in =  instance2.RK_4(r_max,r_min, Nr)
    return data["rs"], final_fs, final_gs, coeff

In [ ]:
fout_prime = (data2["Fs_int_out"][:-1] - data2["Fs_int_out"][1:]) / (data2["rs"][:-1] - data2["rs"][1:])
crossings = np.where(fout_prime[:-1] * fout_prime[1:] < 0)

In [ ]:
renorm_test_rs, renorm_test_fs, renorm_test_gs, renorm_test_coeff = wavefunc_renorm2(80, 5, 1, True)

In [ ]:
renorm_test_coeff

In [ ]:
# renorm_test_rs, renorm_test_fs, renorm_test_gs, renorm_test_coeff = wavefunc_renorm(80,3,1,True)

In [ ]:
(8.130911712453576e-14 / 1.60218e-19) - 511000

In [ ]:
renorm_test_rs[2], renorm_test_rs[1], renorm_test_rs[3]

In [ ]:
renorm_test_fs[0:2] / renorm_test_coeff, renorm_test_gs[0:2] / renorm_test_coeff

In [ ]:
plt.scatter(renorm_test_rs, renorm_test_fs / np.sqrt(renorm_test_coeff))
plt.scatter(renorm_test_rs, renorm_test_gs / np.sqrt(renorm_test_coeff))
# plt.plot(renorm_test_rs,-renorm_test_fs/renorm_test_coeff)
# plt.plot(renorm_test_rs,-renorm_test_gs/renorm_test_coeff)
# plt.vlines(2.3794733234348915e-12,-1e8,1e8,color='black')
plt.xscale("log")
# plt.yscale('log')
# plt.ylim(-500,500)
plt.show()

In [ ]:
generatortest1F = np.load("Fs_80_5_1.npy")
generatortest1G = np.load("Gs_80_5_1.npy")
generatortest1R = np.load("Rs_80_5_1.npy")

In [ ]:
plt.scatter(generatortest1R, generatortest1F)
plt.scatter(generatortest1R, generatortest1G)
# plt.plot(renorm_test_rs,-renorm_test_fs/renorm_test_coeff)
# plt.plot(renorm_test_rs,-renorm_test_gs/renorm_test_coeff)
# plt.vlines(2.3794733234348915e-12,-1e8,1e8,color='black')
plt.xscale("log")
# plt.yscale('log')
# plt.ylim(-500,500)
plt.show()